# Phase 2C: paired counterfactual training (Kaggle GPU)

This notebook trains the **existing** Phase 2A GRU and Phase 2B graph architectures on the Phase 2C branch data (`scripts/train_paired.py`, `configs/phase2c/paired_training.json`). Each run has the same LOBO fold, normalizer, windows, Huber state loss, optimizer, schedule and early stopping as Phase 2A/2B.

| Objective | Loss |
|---|---|
| `arms` (data-only control) | L_state(Phase 1) + L_state(branch arms as ordinary trajectories) |
| `paired` | the same + λ · L_cf (autoregressive nominal vs intervention arms from the same context; Δ trajectory loss over all 36 steps) |

Paired vs `arms` isolates the counterfactual objective. The baselines A (GRU, state loss) and C (graph, state loss) are the existing Phase 2A/2B checkpoints.

**Stages** (one Kaggle session each is a safe assumption; finished jobs are skipped on rerun):
1. λ sweep on `sweep_fold` (training/validation branch points only) + `arms` controls → automatic λ selection (rule in config).
2. `arms` controls on `control_folds`; `paired` on all 6 folds for GRU and graph.
3. Graph `paired` ensemble seeds 1–2 on all folds.

**Kaggle settings:** GPU (T4 ×2 or P100), Internet on.

**Inputs:**
- the dataset with `phase1_sample.zip`;
- the **output of `phase2c_collect_counterfactual`** (its `phase2c_counterfactual/`);
- for sessions 2+, this notebook's previous version (for resume).

**Output:** `/kaggle/working/phase2c_training/` + `phase2c_training.zip`. Unzip at the repo root (the zip contains `results/phase2c/training/...` and `configs/phase2c/split_manifest.json`).

In [ ]:
import os, sys, glob, json, shutil, subprocess, time
BRANCH = "phase2c"
STARTER_KIT_REV = "1fb75606d12e016463fbc63de704967c5b2bf9d5"
STAGES = [1]               # run [1] in session 1, [2] in session 2, [3] in session 3 (or several, e.g. [1, 2])
PARALLEL = 3               # concurrent jobs (2 GPUs -> round-robin); data loading is CPU-bound (4 cores)
DEVICE = "cuda"
LAMBDA = None              # None: select automatically from the stage-1 sweep; or force a value, e.g. 0.3
SWEEP_FOLD = None          # None: config value
HELDOUT_WEATHER = "denver" # Test-B scenarios (never trained on)
SMOKE = False              # True: tiny runs (1-2 epochs, few windows) to check the pipeline end to end
FOLDS = [0, 1, 2, 3, 4, 5] # LOBO folds for stage 2/3
CONTROL_FOLDS = None       # None: config value

os.environ["MPLBACKEND"] = "Agg"       # the kernel's inline backend is not importable in the /tmp venv subprocesses
LOCAL_TEST = os.environ.get("P2C_LOCAL_TEST")
if LOCAL_TEST:
    REPO = os.environ["P2C_REPO"]; PY = os.environ["P2C_PY"]; WORK_OUT = os.environ["P2C_OUT"]; INPUT_ROOT = os.environ["P2C_INPUT"]
else:
    REPO = "/tmp/p2c/smart-building-rl"; PY = "/tmp/p2c/venv/bin/python"; WORK_OUT = "/kaggle/working"; INPUT_ROOT = "/kaggle/input"
SCRATCH = os.path.join(WORK_OUT, "_scratch") if LOCAL_TEST else "/tmp/p2c"   # NOT persisted on Kaggle (no symlinks/data copies in output)
OUT = os.path.join(WORK_OUT, "phase2c_training")
os.makedirs(OUT, exist_ok=True)
print("REPO", REPO, "OUT", OUT, "STAGES", STAGES, "SMOKE", SMOKE)


In [ ]:
# ---- 1. environment ----------------------------------------------------------
if not LOCAL_TEST:
    !mkdir -p /tmp/p2c && rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/fannam/smart-building-rl.git {REPO}
    !pip install -q uv
    !uv venv -q --python 3.12 /tmp/p2c/venv
    !git clone -q https://github.com/smart-buildings-2026/starter-kit.git {REPO}/starter-kit && cd {REPO}/starter-kit && git checkout -q {STARTER_KIT_REV}
    !uv pip install -q -p {PY} -e {REPO}/starter-kit
    !uv pip install -q -p {PY} "numpy==2.0.2" "pandas==2.3.3" "torch>=2.4"
!cd {REPO} && git log --oneline -1
!{PY} -c "import torch; print(torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count())"


In [ ]:
# ---- 2. data: Phase 1 windows + Phase 2C branches; QA + split manifest ---------
def find(name, marker):
    for d in glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True):
        if os.path.exists(os.path.join(d, marker)):
            return d
    return None
p1 = f"{REPO}/data/phase1_sample"
if not os.path.exists(f"{p1}/manifest.jsonl"):
    os.makedirs(f"{REPO}/data", exist_ok=True)
    d = find("phase1_sample", "manifest.jsonl")
    if d:
        os.symlink(d, p1)
    else:
        z = glob.glob(f"{INPUT_ROOT}/**/phase1_sample.zip", recursive=True)
        assert z, "attach the dataset with phase1_sample.zip"
        subprocess.run(["unzip", "-q", z[0], "-d", f"{REPO}/data"], check=True)
assert sum(1 for _ in open(f"{p1}/manifest.jsonl")) == 60
CF = f"{SCRATCH}/phase2c_counterfactual"
os.makedirs(SCRATCH, exist_ok=True)
if not os.path.exists(f"{CF}/plan.json"):
    d = find("phase2c_counterfactual", "plan.json")
    if d:
        os.symlink(d, CF)
    else:
        z = glob.glob(f"{INPUT_ROOT}/**/phase2c_counterfactual.zip", recursive=True)
        assert z, "attach the OUTPUT of phase2c_collect_counterfactual (phase2c_counterfactual/ or its zip)"
        subprocess.run(["unzip", "-q", z[0], "-d", SCRATCH], check=True)
QA_DIR = f"{SCRATCH}/collection_qa"; os.makedirs(QA_DIR, exist_ok=True)   # writable view of the (read-only) input
shutil.copy2(f"{CF}/plan.json", QA_DIR)
if not os.path.islink(f"{QA_DIR}/branches"):
    shutil.rmtree(f"{QA_DIR}/branches", ignore_errors=True); os.symlink(os.path.realpath(f"{CF}/branches"), f"{QA_DIR}/branches")
qa = subprocess.run(f"cd {REPO} && {PY} scripts/phase2c_collection_qa.py --out {QA_DIR}", shell=True, capture_output=True, text=True)
print(qa.stdout.splitlines()[0] if qa.stdout else qa.stderr[-2000:])
rep = json.load(open(f"{QA_DIR}/qa_report.json"))
assert not rep["hard_failures"], rep["hard_failures"][:5]
if not rep["all_planned_complete"]:
    print("WARNING: collection incomplete -> training uses the branch points that exist", rep["incomplete_scenarios"])
os.makedirs(f"{OUT}/collection_qa", exist_ok=True)
for f in ("qa_report.json", "manifest.jsonl"):
    shutil.copy2(f"{QA_DIR}/{f}", f"{OUT}/collection_qa/")
SPLITS = f"{OUT}/split_manifest.json"
!cd {REPO} && {PY} scripts/phase2c_splits.py --data {QA_DIR} --out {SPLITS} --heldout-weather {HELDOUT_WEATHER}
!mkdir -p {REPO}/configs/phase2c && cp {SPLITS} {REPO}/configs/phase2c/split_manifest.json


In [ ]:
# ---- 3. run config + resume from a previous session --------------------------
cfg = json.load(open(f"{REPO}/configs/phase2c/paired_training.json"))
cfg.update(cf_data=CF, split_manifest=SPLITS, phase1_root=p1, cache=f"{SCRATCH}/cache")
if CONTROL_FOLDS is not None:
    cfg["control_folds"] = CONTROL_FOLDS
if SMOKE:
    cfg["training_overrides"] = {"epochs": 2, "max_train_windows": 64, "val_stride": 60, "min_epochs": 0}
if SWEEP_FOLD is not None:
    cfg["sweep_fold"] = SWEEP_FOLD
RUN_CFG = f"{OUT}/run_config.json"; json.dump(cfg, open(RUN_CFG, "w"), indent=1)
TRAIN_OUT = f"{OUT}/results/phase2c/training"
os.makedirs(TRAIN_OUT, exist_ok=True)
n_imp = 0
for prev in glob.glob(f"{INPUT_ROOT}/**/phase2c_training/results/phase2c/training", recursive=True):
    if os.path.realpath(prev) == os.path.realpath(TRAIN_OUT):
        continue
    for f in glob.glob(f"{prev}/**/*", recursive=True):
        if os.path.isfile(f):
            t = os.path.join(TRAIN_OUT, os.path.relpath(f, prev))
            if not os.path.exists(t):
                os.makedirs(os.path.dirname(t), exist_ok=True); shutil.copy2(f, t); n_imp += 1
    for f in ("selected_lambda.json",):
        s = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(prev))), f)
        if os.path.exists(s) and not os.path.exists(f"{OUT}/{f}"):
            shutil.copy2(s, OUT)
print("imported files from previous sessions:", n_imp)
done = sorted(glob.glob(f"{TRAIN_OUT}/*/*/train_log_seed*.json"))
print("finished jobs present:", len(done))

def run(jobs, tag):
    jobs = [j for j in jobs]
    if not jobs:
        return
    log = f"{OUT}/train_{tag}_{int(time.time())}.log"
    print(f"{tag}: {len(jobs)} jobs -> {log}")
    cmd = (f"cd {REPO} && {PY} -u scripts/train_paired.py --config {RUN_CFG} --out {TRAIN_OUT} --parallel {PARALLEL} "
           f"--device {DEVICE} --workers 1 --threads 1 --jobs {' '.join(jobs)} 2>&1 | grep -v -i warning | tee {log} | grep -E 'done|skip|FAIL'")
    subprocess.run(cmd, shell=True)


In [ ]:
# ---- 4. GPU smoke test (1 tiny job into /tmp; stops here if the CUDA path is broken) ----
sm = dict(cfg, training_overrides={"epochs": 1, "max_train_windows": 32, "val_stride": 60, "min_epochs": 0})
json.dump(sm, open(f"{SCRATCH}/smoke_cfg.json", "w"))
fold_smoke = cfg["sweep_fold"]
r = subprocess.run(f"cd {REPO} && {PY} scripts/train_paired.py --config {SCRATCH}/smoke_cfg.json --out {SCRATCH}/smoke_train --parallel 1 "
                   f"--device {DEVICE} --workers 1 --threads 1 --jobs graph:paired:{fold_smoke}:9:0.3", shell=True, capture_output=True, text=True)
print(r.stdout[-1500:]); assert "done graph:paired" in r.stdout or "skip" in r.stdout, r.stderr[-3000:]


In [ ]:
# ---- 5. STAGE 1: lambda sweep on the sweep fold + automatic selection ----------
F = cfg["sweep_fold"]
if 1 in STAGES:
    run([f"{a}:arms:{F}:0:0" for a in ("gru", "graph")] +
        [f"{a}:paired:{F}:0:{l:g}" for a in ("gru", "graph") for l in cfg["lambda_grid"]], "stage1")

def select_lambda(arch):
    def log_of(obj, lam):
        p = glob.glob(f"{TRAIN_OUT}/{arch}_{obj}_lam{lam:g}/fold_{F}_*/train_log_seed0.json")
        return json.load(open(p[0])) if p else None
    ctl = log_of("arms", 0)
    cands = [(l, log_of("paired", l)) for l in cfg["lambda_grid"]]
    cands = [(l, L) for l, L in cands if L is not None]
    if ctl is None or not cands:
        return None, "sweep incomplete"
    if ctl["val_branch_points"] == 0:
        return None, "no validation branch points on the sweep fold -> cannot select lambda (set LAMBDA explicitly)"
    ok = [(l, L) for l, L in cands if L["best_val_state"] <= 1.05 * ctl["best_val_state"]]
    pool = ok or cands
    l, L = min(pool, key=lambda x: x[1]["best_val_cf"])
    table = {f"{x:g}": {"val_state": Y["best_val_state"], "val_cf": Y["best_val_cf"]} for x, Y in cands}
    return l, {"control_val_state": ctl["best_val_state"], "control_val_cf": ctl["best_val_cf"], "candidates": table,
               "within_state_tolerance": [f"{x:g}" for x, _ in ok], "rule": cfg["lambda_selection_rule"]}
sel_path = f"{OUT}/selected_lambda.json"
sel = json.load(open(sel_path)) if os.path.exists(sel_path) else {}
for arch in ("gru", "graph"):
    if LAMBDA is not None:
        sel[arch] = {"lambda": LAMBDA, "why": "forced by notebook parameter"}
    elif arch not in sel:
        l, why = select_lambda(arch)
        if l is not None:
            sel[arch] = {"lambda": l, "why": why}
        else:
            print(f"lambda NOT selected for {arch}: {why}")
json.dump(sel, open(sel_path, "w"), indent=1)
print(json.dumps(sel, indent=1))


In [ ]:
# ---- 6. STAGE 2: controls on control folds + paired on all 6 folds ------------
if 2 in STAGES:
    assert all(a in sel for a in ("gru", "graph")), "run stage 1 first (or set LAMBDA)"
    jobs = [f"{a}:arms:{f}:0:0" for a in ("gru", "graph") for f in cfg["control_folds"]]
    jobs += [f"{a}:paired:{f}:0:{sel[a]['lambda']:g}" for a in ("gru", "graph") for f in FOLDS]
    run(jobs, "stage2")


In [ ]:
# ---- 7. STAGE 3: graph paired ensemble (extra seeds) ---------------------------
if 3 in STAGES:
    assert "graph" in sel, "run stage 1 first (or set LAMBDA)"
    run([f"graph:paired:{f}:{s}:{sel['graph']['lambda']:g}" for s in cfg["ensemble_seeds"] for f in FOLDS], "stage3")


In [ ]:
# ---- 8. summary + package ---------------------------------------------------------
rows = []
for p in sorted(glob.glob(f"{TRAIN_OUT}/*/*/train_log_seed*.json")):
    L = json.load(open(p))
    rows.append((L["spec"], L["best_epoch"], L["epochs_run"], round(L["best_val_state"], 4), round(L["best_val_cf"], 4),
                 L["train_branch_points"], L["device"], round(L["train_s"] / 60, 1)))
print(f"{'job':28s} best/ep  val_state  val_cf  branches device  min")
for r in rows:
    print(f"{r[0]:28s} {r[1]:>3}/{r[2]:<3} {r[3]:9.4f} {r[4]:7.4f} {r[5]:8d} {r[6]:6s} {r[7]}")
shutil.copy2(SPLITS, f"{OUT}/results/phase2c/split_manifest.json")
!cd {OUT} && rm -f {WORK_OUT}/phase2c_training.zip && zip -qr {WORK_OUT}/phase2c_training.zip results selected_lambda.json split_manifest.json run_config.json collection_qa/qa_report.json collection_qa/manifest.jsonl
!ls -lh {WORK_OUT}/phase2c_training.zip
